## Collaborative Filtering

This notebook trains an ALS (Alternating Least Squares) model with weighted confidence for implicit feedback.  
The goal is to learn item embeddings that capture collaborative patterns ("players who played X also played Y") from steam user-game interactions.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import sparse
from implicit.als import AlternatingLeastSquares

PROJECT_ROOT = Path.cwd().parent

df = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "recommendations.csv",
    usecols=["user_id", "app_id", "is_recommended", "hours"]
)

print(f"Total interactions: {len(df):,}")
print(f"Unique users: {df['user_id'].nunique():,}")
print(f"Unique games: {df['app_id'].nunique():,}")

/home/axel/grec/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total interactions: 41,154,794
Unique users: 13,781,059
Unique games: 37,610


### Data Cleaning

Remove zero-hour interactions (no real engagement) and duplicate user-game pairs.

In [2]:
df = df[df["hours"] > 0]
df = df[df["is_recommended"] == True]
df = df.drop_duplicates(subset=["user_id", "app_id"])

print(f"After cleaning: {len(df):,} interactions") 

After cleaning: 35,198,680 interactions


### Iterative K-Core Filtering

Single-pass filtering (remove users < 10, games < 100) leaves orphans: after removing games, some users drop below the threshold again.  
Iterative k-core repeats until both constraints hold simultaneously, producing a denser and cleaner matrix.

In [3]:
MIN_USER_INTERACTIONS = 10
MIN_GAME_INTERACTIONS = 100

prev_len = 0
iteration = 0

while len(df) != prev_len:
    prev_len = len(df)
    iteration += 1

    user_counts = df["user_id"].value_counts()
    df = df[df["user_id"].isin(user_counts[user_counts >= MIN_USER_INTERACTIONS].index)]

    game_counts = df["app_id"].value_counts()
    df = df[df["app_id"].isin(game_counts[game_counts >= MIN_GAME_INTERACTIONS].index)]

    print(f"  Iteration {iteration}: {len(df):,} interactions, {df['user_id'].nunique():,} users, {df['app_id'].nunique():,} games")

num_users = df["user_id"].nunique()
num_games = df["app_id"].nunique()
sparsity = 1 - len(df) / (num_users * num_games)

print(f"\nFinal: {len(df):,} interactions | {num_users:,} users | {num_games:,} games | Sparsity: {sparsity:.4f}")

  Iteration 1: 10,548,001 interactions, 545,926 users, 8,081 games
  Iteration 2: 10,337,962 interactions, 522,986 users, 7,902 games
  Iteration 3: 10,329,442 interactions, 522,145 users, 7,892 games
  Iteration 4: 10,329,060 interactions, 522,100 users, 7,892 games
  Iteration 5: 10,329,060 interactions, 522,100 users, 7,892 games

Final: 10,329,060 interactions | 522,100 users | 7,892 games | Sparsity: 0.9975


### Feature Engineering

- `hours_log`: log-transformed playtime, outliers capped at 99th percentile  
- `confidence`: Hu et al. (2008) formulation — `1 + α × hours_log`. Higher playtime = higher confidence that the user truly likes the game

In [4]:
ALPHA = 20

df["hours_log"] = np.log1p(df["hours"].clip(upper=df["hours"].quantile(0.99)))
df_cf = df[["user_id", "app_id", "hours_log"]].copy()
df_cf["confidence"] = 1 + ALPHA * df_cf["hours_log"]

print(df_cf.head(10))
print(f"\nConfidence range: {df_cf['confidence'].min():.1f} - {df_cf['confidence'].max():.1f}") 

     user_id  app_id  hours_log  confidence
22     22793  534380   3.728100   75.562003
23    271318  518790   2.397895   48.957905
32    912612  438100   2.208274   45.165488
41    763450  359550   5.121580  103.431604
51    461080     730   3.109061   63.181219
58    737481  602960   3.758872   76.177437
80   1199958  489830   5.560682  112.213633
89    628967  620980   3.526361   71.527210
111  2832559  238960   2.468100   50.361991
118  3473951  960090   3.693867   74.877340

Confidence range: 2.9 - 133.1


### Sparse Matrix Construction

ALS expects a sparse `user × item` matrix. The values are confidence-weighted preferences.

In [5]:
# Map original IDs to contiguous indices
user_ids = df_cf["user_id"].unique()
item_ids = df_cf["app_id"].unique()

user_id_to_idx = {uid: idx for idx, uid in enumerate(user_ids)}
item_id_to_idx = {iid: idx for idx, iid in enumerate(item_ids)}
idx_to_item_id = {idx: int(iid) for iid, idx in item_id_to_idx.items()}
idx_to_user_id = {idx: int(uid) for uid, idx in user_id_to_idx.items()}

rows = df_cf["user_id"].map(user_id_to_idx).values
cols = df_cf["app_id"].map(item_id_to_idx).values
values = df_cf["confidence"].values

user_item = sparse.csr_matrix(
    (values, (rows, cols)),
    shape=(len(user_ids), len(item_ids)),
)

print(f"User-Item matrix: {user_item.shape}")
print(f"Non-zero entries: {user_item.nnz:,}")
print(f"Density: {user_item.nnz / (user_item.shape[0] * user_item.shape[1]):.6f}")

User-Item matrix: (522100, 7892)
Non-zero entries: 10,329,060
Density: 0.002507


### Train / Test Split

In [6]:
from implicit.evaluation import train_test_split

train_matrix, test_matrix = train_test_split(user_item, train_percentage=0.8, random_state=42) 

### ALS Training

Implict's ALS alternates between fixing user factors and solving for item factors, then vice versa. The confidence values tell the model how much to trust each observation.

In [7]:
model = AlternatingLeastSquares(
    factors=128,
    iterations=80,
    regularization=0.01,
    random_state=42,
)

model.fit(train_matrix)

/home/axel/grec/.venv/lib/python3.12/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 16 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 80/80 [02:48<00:00,  2.10s/it]


### Evaluation

- **Precision@10**: of the top 10 recommended items, what fraction appear in the held-out test set?  
- **MAP@10**: same idea but rewards models that rank test items higher in the list

In [8]:
K = 10
N_EVAL_USERS = 50_000  # sample for speed

rng = np.random.default_rng(42)
test_users = np.array(list(set(test_matrix.nonzero()[0])))
eval_users = rng.choice(test_users, size=min(N_EVAL_USERS, len(test_users)), replace=False)

precisions = []
avg_precisions = []

for user_idx in eval_users:
    
    test_items = set(test_matrix[user_idx].nonzero()[1])
    if not test_items:
        continue

    rec_indices, _ = model.recommend(
        user_idx, train_matrix[user_idx], N=K, filter_already_liked_items=True
    )

    # Precision@K
    hits = [1 if r in test_items else 0 for r in rec_indices]
    precisions.append(sum(hits) / K)

    # Average Precision@K
    cum_hits = 0
    ap = 0.0
    for rank, h in enumerate(hits, 1):
        cum_hits += h
        if h:
            ap += cum_hits / rank
    avg_precisions.append(ap / min(len(test_items), K))

print(f"Evaluated on {len(precisions):,} users")
print(f"Precision@{K}: {np.mean(precisions):.4f}")
print(f"MAP@{K}:       {np.mean(avg_precisions):.4f}")

Evaluated on 50,000 users
Precision@10: 0.0506
MAP@10:       0.0607


Although the obtained results may initially seem low, they are actually reasonable given the evaluation setup. Most cases, user has only 2 relevant items in the test set (with 8 used for training), which imposes a strict upper bound of 0.2 on Precision@10, since at most 2 relevant items can appear in the top 10 recommendations. This means the model is achieving roughly a quarter of the maximum possible score, which indicates it is capturing useful patterns rather than performing poorly.

### Sample Recommendations

Quick sanity check: pick a random user and see what the model recommends vs what they actually played.

In [11]:
# Load game names lookup
games_raw = pd.read_json(PROJECT_ROOT / "data" / "raw" / "games.json", orient="index")
app_id_to_name = games_raw["name"].to_dict()  # {app_id_int: "Game Name"}

sample_user_idx = np.random.randint(0, len(user_ids))
sample_user_id = idx_to_user_id[sample_user_idx]

# What they already played
played_indices = user_item[sample_user_idx].nonzero()[1]
played_app_ids = [idx_to_item_id[i] for i in played_indices]
played_names = [app_id_to_name.get(app_id, f"Unknown ({app_id})") for app_id in played_app_ids]
print(f"User {sample_user_id} played {len(played_app_ids)} games:")
for name in played_names:
    print(f"  {name}")

# Model recommendations
rec_indices, rec_scores = model.recommend(
    sample_user_idx,
    user_item[sample_user_idx],
    N=10,
    filter_already_liked_items=True,
)

print("\nTop 10 recommendations:")
for idx, score in zip(rec_indices, rec_scores):
    app_id = idx_to_item_id[idx]
    name = app_id_to_name.get(app_id, f"Unknown ({app_id})")
    print(f"  {name:<50}  score={score:.4f}")


User 3859710 played 10 games:
  Dying Light 2 Stay Human: Reloaded Edition
  Tom Clancy's Rainbow Six® Siege X
  BioShock Infinite
  Far Cry® 5
  DEVOUR
  Aseprite
  Paint the Town Red
  Mirror's Edge™ Catalyst
  Rage Wars
  Seen

Top 10 recommendations:
  Dying Light                                         score=1.0825
  Cyberpunk 2077                                      score=0.8692
  Wallpaper Engine                                    score=0.6856
  Dishonored                                          score=0.6802
  Dishonored 2                                        score=0.6593
  Teardown                                            score=0.5768
  People Playground                                   score=0.5757
  Red Dead Redemption 2                               score=0.5168
  Retrowave                                           score=0.4954
  Black Mesa                                          score=0.4861


### Find related games 

In [10]:
from scipy import sparse
 
liked_app_ids = [1110910, 570940, 1245620, 485510, 582010]
liked_hours   = [20, 67, 120, 60, 80] 

ALPHA = 30
 
print("=== New user's library ===")
for app_id, hours in zip(liked_app_ids, liked_hours):
    name = app_id_to_name.get(int(app_id), f"Unknown ({app_id})")
    print(f"  {name:<50}  hours = {hours}")
 
n_items = len(item_ids)
cols, vals = [], []
 
for app_id, hours in zip(liked_app_ids, liked_hours):
    if app_id not in item_id_to_idx:
        print(f"  ⚠  app_id {app_id} not in training catalogue — skipped")
        continue
    item_idx = item_id_to_idx[app_id]
    hours_log  = np.log1p(min(hours, df["hours"].quantile(0.99)))
    confidence = 1 + ALPHA * hours_log
    cols.append(item_idx)
    vals.append(confidence)
 
new_user_items = sparse.csr_matrix(
    (vals, ([0] * len(cols), cols)),
    shape=(1, n_items),
)
 

rec_indices, rec_scores = model.recommend(
    userid=0,                        # placeholder; ignored when recalculating
    user_items=new_user_items,
    N=10,
    filter_already_liked_items=True,
    recalculate_user=True,           
)
 
print("\n=== Top-10 recommendations for the new user ===")
for rank, (idx, score) in enumerate(zip(rec_indices, rec_scores), 1):
    app_id = idx_to_item_id[idx]
    name   = app_id_to_name.get(app_id, f"Unknown ({app_id})")
    print(f"  {rank:>2}. {name:<50}  score = {score:.4f}")

=== New user's library ===
  Mortal Shell                                        hours = 20
  DARK SOULS™: REMASTERED                             hours = 67
  ELDEN RING                                          hours = 120
  Nioh: Complete Edition                              hours = 60
  Monster Hunter: World                               hours = 80

=== Top-10 recommendations for the new user ===
   1. DARK SOULS™ II: Scholar of the First Sin            score = 0.9930
   2. DARK SOULS™ III                                     score = 0.8488
   3. Nioh 2 – The Complete Edition                       score = 0.8431
   4. Sekiro™: Shadows Die Twice - GOTY Edition           score = 0.7555
   5. CODE VEIN                                           score = 0.7090
   6. Remnant: From the Ashes                             score = 0.6977
   7. The Surge                                           score = 0.6554
   8. Salt and Sanctuary                                  score = 0.6304
   9. Blasphem

The recommendation results demonstrate that the model is effectively capturing the user’s preferences and generalizing them to similar items. The user’s library is strongly centered around challenging action RPGs and “soulslike” games, and the top-10 recommendations closely align with this pattern, prominently featuring highly relevant titles such as Dark Souls II, Dark Souls III, Nioh 2, and Sekiro. Additionally, the inclusion of slightly more diverse but still related games (e.g., CODE VEIN, Remnant, The Surge) suggests the system is not merely memorizing exact matches but also exploring nearby items in the latent space.